In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import dask.dataframe as dd
from dask import delayed
from dask.diagnostics import ProgressBar
from pandas.plotting import register_matplotlib_converters
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from joblib import Parallel, delayed, Memory, cpu_count
from dtaidistance import dtw
from sklearn.cluster import MiniBatchKMeans
from fastdtw import fastdtw
from umap import UMAP

In [ ]:
memory = Memory(location='cachedir', verbose=0)

In [ ]:
cpu_count()

In [ ]:
# 手动注册 Matplotlib 转换器
register_matplotlib_converters()

# 项目背景

# 数据导入

In [ ]:
@memory.cache
def read_hhblock_dataset(file_path):
    # 获取文件夹内所有CSV文件的路径
    csv_files = [os.path.join(file_path, file) for file in os.listdir(file_path) if file.endswith('.csv')]

    # 读取每个CSV文件为DataFrame，并存入列表
    dataframes = [pd.read_csv(file) for file in csv_files]

    # 合并所有DataFrame为一个大的DataFrame
    hhblock_big_dataframe = pd.concat(dataframes, ignore_index=True)

    return hhblock_big_dataframe

In [ ]:
@memory.cache
def data_import():
    informations_households = pd.read_csv('archive/informations_households.csv')
    daily_dataset = pd.read_csv('archive/daily_dataset.csv')
    halfhourly_block_0 = pd.read_csv('archive/halfhourly_dataset/halfhourly_dataset/block_0.csv')
    hhblock_0 = pd.read_csv('archive/hhblock_dataset/hhblock_dataset/block_0.csv')
    hhblock_big_dataframe = read_hhblock_dataset('archive\hhblock_dataset\hhblock_dataset')
    uk_bank_holidays = pd.read_csv('archive/uk_bank_holidays.csv')
    weather_daily_darksky = pd.read_csv('archive/weather_daily_darksky.csv')
    weather_hourly_darksky = pd.read_csv('archive/weather_hourly_darksky.csv')
    return informations_households, daily_dataset, halfhourly_block_0, hhblock_0, hhblock_big_dataframe, uk_bank_holidays, weather_daily_darksky, weather_hourly_darksky

In [ ]:
%%time
informations_households, daily_dataset, halfhourly_block_0, hhblock_0, hhblock_big_dataframe, uk_bank_holidays, weather_daily_darksky, weather_hourly_darksky = data_import()

In [ ]:
# %prun -s cumulative -l 10 informations_households, daily_dataset, halfhourly_block_0, hhblock_0, hhblock_big_dataframe, uk_bank_holidays, weather_daily_darksky, weather_hourly_darksky = data_import()

# 解释性描述数据与可视化
- 数据查看与理解
- 数据清洗
- 分组可视化

In [ ]:
GLOBAL_PLOT_LEVEL = 0  # 0: 不显示图形，1: 显示部分图形，2: 显示所有图形
def plot_level(level, function):
    if GLOBAL_PLOT_LEVEL >= level:
        function()

## informations_households.csv
informations_households.csv - 包含有关每个家庭的信息，如家庭ID、家庭类型、家庭的总电量消耗等。
数据查看与清洗

In [ ]:
informations_households

In [ ]:
display(informations_households.nunique())

In [ ]:
display(informations_households['Acorn'].value_counts())

In [ ]:
display(informations_households['Acorn_grouped'].value_counts())

In [ ]:
display(informations_households['file'].value_counts())

In [ ]:
informations_households_filtered = informations_households[
    (informations_households['Acorn'] != 'ACORN-U') & (informations_households['Acorn'] != 'ACORN-')]
informations_households_filtered

In [ ]:
display(informations_households_filtered['Acorn'].value_counts())

In [ ]:
display(informations_households_filtered['Acorn_grouped'].value_counts())

In [ ]:
valid_lclid = informations_households_filtered['LCLid'].unique()
display(pd.Series(valid_lclid))

In [ ]:
unique_acorn_values = informations_households_filtered[informations_households_filtered['Acorn_grouped'] == 'Affluent'][
    'Acorn'].unique()
display(pd.Series(unique_acorn_values))

In [ ]:
unique_acorn_values = \
    informations_households_filtered[informations_households_filtered['Acorn_grouped'] == 'Comfortable'][
        'Acorn'].unique()
display(pd.Series(unique_acorn_values))

In [ ]:
unique_acorn_values = \
    informations_households_filtered[informations_households_filtered['Acorn_grouped'] == 'Adversity']['Acorn'].unique()
display(pd.Series(unique_acorn_values))

In [ ]:
missing_informations_households_values = informations_households_filtered.isnull().sum()
display(missing_informations_households_values)

## daily_dataset

### 数据查看与清洗

In [ ]:
daily_dataset.head(1000)

In [ ]:
@ memory.cache
def process_daily_dataset(df):
    # 将'day'列转换为日期时间格式
    df['day'] = pd.to_datetime(df['day'])

    # 将'LCLid'列转换为分类数据类型
    df['LCLid'] = df['LCLid'].astype('category')
    
    # 清洗掉没有Acorn信息的数据
    df = df[df['LCLid'].isin(valid_lclid)]

    return df

In [ ]:
filtered_daily_dataset = process_daily_dataset(daily_dataset)
filtered_daily_dataset.head(1000)

In [ ]:
filtered_daily_dataset.info()

### 缺失值处理

In [ ]:
missing_filtered_daily_dataset_values = filtered_daily_dataset.isnull().sum()
display(missing_filtered_daily_dataset_values)

In [ ]:
rows_with_missing_values_specific_columns = filtered_daily_dataset[
    filtered_daily_dataset[['energy_std']].isnull().any(axis=1)]
rows_with_missing_values_specific_columns

In [ ]:
rows_with_missing_values_specific_columns = filtered_daily_dataset[
    filtered_daily_dataset[['energy_median', 'energy_mean']].isnull().any(axis=1)]
rows_with_missing_values_specific_columns

### 分组可视化

In [ ]:
def plot_column_histogram(df, column_name):
    plt.figure(figsize=(10, 6))
    sns.histplot(df[column_name], kde=True)
    plt.title(f'Histogram of {column_name}')
    plt.xlabel(column_name)
    plt.ylabel('Frequency')
    plt.show()

In [ ]:
plot_level(1, lambda: plot_column_histogram(filtered_daily_dataset, 'energy_median'))

In [ ]:
columns_all = ['energy_median', 'energy_mean', 'energy_max', 'energy_std', 'energy_sum', 'energy_min']
columns_without_sum = ['energy_median', 'energy_mean', 'energy_max', 'energy_std', 'energy_min']
columns_without_min = ['energy_median', 'energy_mean', 'energy_max', 'energy_std', 'energy_sum']

In [ ]:
def plot_column_histograms(df, column_names):
    plt.figure(figsize=(10, 6))
    for column_name in column_names:
        sns.histplot(df[column_name], kde=True, label=column_name)
    plt.title(f'Histogram of {" vs ".join(column_names)}')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.xscale('log')  # 使用对数尺度
    plt.legend()
    plt.show()

In [ ]:
plot_level(1, lambda: plot_column_histograms(filtered_daily_dataset, columns_all))
plot_level(1, lambda: plot_column_histograms(filtered_daily_dataset, columns_without_sum))
plot_level(1, lambda: plot_column_histograms(filtered_daily_dataset, columns_without_min))

In [ ]:
def plot_multiple_column_histograms(df, columns):
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    for i, column in enumerate(columns):
        sns.histplot(df[column], kde=True, ax=axes[i])
        axes[i].set_title(f'Histogram of {column}')
        axes[i].set_xlabel(column)
        axes[i].set_ylabel('Frequency')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_level(1, lambda: plot_multiple_column_histograms(filtered_daily_dataset, columns_all))

In [ ]:
def plot_columns_boxplot(df, columns):
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df[columns])
    plt.xticks(rotation=45)
    plt.title('Box plot of Energy Consumption Statistics')
    plt.show()

In [ ]:
plot_level(1, lambda: plot_columns_boxplot(filtered_daily_dataset, columns_all))

In [ ]:
plot_level(1, lambda: plot_columns_boxplot(filtered_daily_dataset, columns_without_sum))

In [ ]:
def plot_column_difference(df, column1, column2):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=column1, y=column2, data=df)
    plt.title(f'Scatter plot of {column1} vs. {column2}')
    plt.xlabel(column1)
    plt.ylabel(column2)
    plt.show()

In [ ]:
plot_level(1, lambda: plot_column_difference(filtered_daily_dataset, 'energy_mean', 'energy_median'))

In [ ]:
plot_level(1, lambda: plot_column_difference(filtered_daily_dataset, 'energy_mean', 'energy_max'))

In [ ]:
plot_level(1, lambda: plot_column_difference(filtered_daily_dataset, 'energy_mean', 'energy_std'))

In [ ]:
plot_level(1, lambda: plot_column_difference(filtered_daily_dataset, 'energy_mean', 'energy_sum'))

In [ ]:
plot_level(1, lambda: plot_column_difference(filtered_daily_dataset, 'energy_mean', 'energy_min'))

In [ ]:
plot_level(1, lambda: plot_column_difference(filtered_daily_dataset, 'energy_median', 'energy_max'))

In [ ]:
plot_level(1, lambda: plot_column_difference(filtered_daily_dataset, 'energy_median', 'energy_count'))

In [ ]:
def plot_yearly_energy_trend(df, stat_col='energy_mean'):
    # 设置图形的大小
    plt.figure(figsize=(14, 8))

    # 获取年份列表
    years = df['day'].dt.year.unique()
    for year in sorted(years):
        # 选择当前年份的数据
        df_year = df[df['day'].dt.year == year]

        # 按'day'分组并计算每一天的平均用电量
        daily_avg = df_year.groupby('day')[stat_col].mean()

        # 重置索引，确保能够按时间序列绘图
        daily_avg = daily_avg.reset_index()

        # 绘制当前年份的日平均用电量趋势
        plt.plot(daily_avg['day'], daily_avg[stat_col], label=str(year))

    # 设置图例
    plt.legend(title="Year")

    # 设置标题和坐标轴标签
    plt.title(f'Yearly Trend of Daily Average {stat_col.capitalize()}')
    plt.xlabel('Day of Year')
    plt.ylabel(f'Average {stat_col.capitalize()}')

    # 显示网格
    plt.grid(True)

    # 显示图形
    plt.show()

# 假设filtered_daily_dataset是你的DataFrame
# plot_yearly_energy_trend(filtered_daily_dataset)


In [ ]:
plot_level(1, lambda: plot_yearly_energy_trend(filtered_daily_dataset))

## halfhourly_dataset

In [ ]:
halfhourly_block_0

In [ ]:
display(halfhourly_block_0.nunique())

In [ ]:
halfhourly_block_0.info()

## hhblock_dataset

### 数据查看与清洗

In [ ]:
hhblock_0.head(1000)

In [ ]:
hhblock_0.info()

In [ ]:
# 查看合并后的DataFrame信息
hhblock_big_dataframe.info()

In [ ]:
hhblock_big_dataframe.head(1000)

In [ ]:
def process_hhblock_dataset(df):
    # 将'day'列转换为日期时间格式
    df['day'] = pd.to_datetime(df['day'])

    # 将'LCLid'列转换为分类数据类型
    df['LCLid'] = df['LCLid'].astype('category')
    
    # 清洗掉没有Acorn信息的数据
    df = df[df['LCLid'].isin(valid_lclid)]

    return df

In [ ]:
filtered_big_hhblock = process_hhblock_dataset(hhblock_big_dataframe)
filtered_big_hhblock.head(1000)

In [ ]:
filtered_big_hhblock.info()

### 缺失值处理

In [ ]:
missing_filtered_big_hhblock_values = filtered_big_hhblock.isnull().sum()
missing_filtered_big_hhblock_values

In [ ]:
@ memory.cache
def data_for_hhblock_plot():
    # 选择所有数值列用于分组聚合
    df_numeric = filtered_big_hhblock.iloc[:, 2:]  # 选择从第三列到最后的所有列
    
    # 按年分组求均值
    df_yearly_mean = df_numeric.groupby(filtered_big_hhblock['day'].dt.to_period('Y')).mean()
    
    # 按季度分组求均值
    df_quarterly_mean = df_numeric.groupby(filtered_big_hhblock['day'].dt.to_period('Q')).mean()
    
    # 按月分组求均值
    df_monthly_mean = df_numeric.groupby(filtered_big_hhblock['day'].dt.to_period('M')).mean()
    
    # 获取季度和月份的字符串表示，不考虑年份
    # 例如，将"2011Q4"转换为"Q4"，将2012-05转换为"M5"
    quarter_str = filtered_big_hhblock['day'].dt.to_period('Q').astype(str).str[-2:]
    month_str = filtered_big_hhblock['day'].dt.month.apply(lambda x: f"M{x}")
    
    # 按季度分组求均值，这里使用转换后的季度字符串
    df_year_quarterly_mean = df_numeric.groupby(quarter_str).mean()
    
    # 按月份分组求均值，这里使用转换后的月份字符串
    df_year_monthly_mean = df_numeric.groupby(month_str).mean()
    
    # 按星期几分组求均值
    df_dayofweek_mean = df_numeric.groupby(filtered_big_hhblock['day'].dt.dayofweek).mean()
    
    return df_yearly_mean, df_quarterly_mean, df_monthly_mean, df_year_quarterly_mean, df_year_monthly_mean, df_dayofweek_mean

In [ ]:
df_yearly_mean, df_quarterly_mean, df_monthly_mean, df_year_quarterly_mean, df_year_monthly_mean, df_dayofweek_mean = data_for_hhblock_plot()

In [ ]:
df_yearly_mean

In [ ]:
df_quarterly_mean

In [ ]:
df_monthly_mean

In [ ]:
df_year_quarterly_mean

In [ ]:
df_year_monthly_mean

In [ ]:
df_dayofweek_mean

In [ ]:
def plot_with_seasonal_colors(df, title_suffix):
    plt.figure(figsize=(14, 8))

    # 定义季节或月份颜色映射
    # 直接通过字符串切片提取出所有唯一的季节/月份
    unique_seasons = sorted({str(index)[-2:] for index in df.index})
    season_colors = plt.cm.viridis(np.linspace(0, 1, len(unique_seasons)))
    color_map = dict(zip(unique_seasons, season_colors))

    # x轴标签
    x_labels = [f'{i // 2:02d}:{i % 2 * 30:02d}' for i in range(48)]

    # 遍历DataFrame的每一行
    for index, row in df.iterrows():
        year, season = str(index)[:-2], str(index)[-2:]
        # 确保row.values中的所有值都是可以绘图的数值类型
        values = [float(value) for value in row.values if isinstance(value, (int, float))]

        # 为这个季节/月份的行选择颜色
        color = color_map[season]

        # 绘制每半小时的平均用电量
        plt.plot(x_labels, values, marker='o', linestyle='-', label=f'{year} {season}', color=color)

    # 自定义图例和图表布局
    plt.legend(title="Time Period", loc='upper right', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.title(f'Average Daily Electricity Consumption per 30 Minutes - {title_suffix}')
    plt.xlabel('Time of Day (in 30-minute intervals)')
    plt.ylabel('Average Electricity Consumption (kWh)')
    plt.xticks(range(0, 48, 3), labels=x_labels[::3])
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_level(1, lambda: plot_with_seasonal_colors(df_yearly_mean, 'Yearly'))

In [ ]:
plot_level(1, lambda: plot_with_seasonal_colors(df_quarterly_mean, 'Quarterly'))

In [ ]:
plot_level(1, lambda: plot_with_seasonal_colors(df_monthly_mean, 'Monthly'))

In [ ]:
plot_level(1, lambda: plot_with_seasonal_colors(df_year_quarterly_mean, 'Yearly Quarterly'))

In [ ]:
plot_level(1, lambda: plot_with_seasonal_colors(df_year_monthly_mean, 'Yearly Monthly'))

In [ ]:
plot_level(1, lambda: plot_with_seasonal_colors(df_dayofweek_mean, 'Day of Week'))

## uk_bank_holidays

### 数据查看与清洗

In [ ]:
uk_bank_holidays

In [ ]:
def process_uk_bank_holidays(df_bank_holidays, df_hhblock):
    # 将'Bank holidays'列转换为日期时间格式
    df_bank_holidays['Bank holidays'] = pd.to_datetime(df_bank_holidays['Bank holidays'])

    # 首先将uk_bank_holidays的日期列转换为日期格式
    df_bank_holidays['Bank holidays'] = pd.to_datetime(df_bank_holidays['Bank holidays'])
    # 创建一个集合，包含所有的假期日期
    holidays_set = set(df_bank_holidays['Bank holidays'])

    # 使用.copy()确保filtered_big_hhblock是一个独立的副本，避免后续修改影响原始数据
    filtered_big_hhblock_copy = df_hhblock.copy()
    
    filtered_big_hhblock_copy.loc[:, 'is_holiday'] = filtered_big_hhblock_copy['day'].isin(holidays_set)
    
    df_numeric = filtered_big_hhblock_copy.iloc[:, 2:]  # 选择数值列
    
    # 根据is_holiday列分组求均值
    df_holidays_mean = df_numeric.groupby(filtered_big_hhblock_copy['is_holiday']).mean()
    
    # 重命名索引以更清晰地表示数据
    df_holidays_mean.index = ['out_holidays', 'in_holidays']

    return df_holidays_mean

In [ ]:
%%time
df_holidays_mean = process_uk_bank_holidays(uk_bank_holidays, filtered_big_hhblock)

In [ ]:
def plot_average_daily_consumption(df, title_suffix):
    plt.figure(figsize=(14, 8))

    # 创建表示24小时内每半小时间隔的x轴标签列表
    x_labels = [f'{i//2:02d}:{i%2*30:02d}' for i in range(48)]

    for index, row in df.iterrows():
        # 转换index为字符串，适用于任何形式的Period或其他索引类型
        index_str = str(index)

        # 确保row.values中的所有值都是可以绘图的数值类型
        # 这里直接使用row.values[:-1]来排除可能的非数值列
        # 假设最后一列是非数值列，如果不是，请根据实际情况调整
        values = [float(value) for value in row.values if isinstance(value, (int, float))]

        # 绘制每半小时的平均用电量
        plt.plot(x_labels, values, marker='o', linestyle='-', label=index_str)

    plt.legend(title="Time Period", loc='upper right')
    plt.title(f'Average Daily Electricity Consumption per 30 Minutes - {title_suffix}')
    plt.xlabel('Time of Day (in 30-minute intervals)')
    plt.ylabel('Average Electricity Consumption (kWh)')
    plt.xticks(range(0, 48, 3), labels=x_labels[::3])
    plt.grid(True)
    plt.show()

In [ ]:
plot_level(1, lambda: plot_average_daily_consumption(df_holidays_mean.iloc[:, :-1], 'Holidays'))

## Acorn数据分析与可视化

In [ ]:
# 合并DataFrame
merged_info_hhblock_df = pd.merge(filtered_big_hhblock,
                                  informations_households_filtered[['LCLid', 'stdorToU', 'Acorn', 'Acorn_grouped']],
                                  on='LCLid', how='left')

In [ ]:
merged_info_hhblock_df.head(1000)

In [ ]:
def data_for_acorn_plot(merged_info_hhblock_df):
    # 选择所有数值列用于分组聚合
    df_numeric = merged_info_hhblock_df.iloc[:, 2:-3]  # 选择从第三列到最后的所有列
    
    # 按'stdorToU'分组求均值
    df_std_tou_mean = df_numeric.groupby(merged_info_hhblock_df['stdorToU']).mean()
    
    # 按'Acorn'分组求均值
    df_acorn_mean = df_numeric.groupby(merged_info_hhblock_df['Acorn']).mean()
    
    # 按'Acorn_grouped'分组求均值
    df_acorn_grouped_mean = df_numeric.groupby(merged_info_hhblock_df['Acorn_grouped']).mean()
    
    return df_std_tou_mean, df_acorn_mean, df_acorn_grouped_mean

In [ ]:
%%time
df_std_tou_mean, df_acorn_mean, df_acorn_grouped_mean = data_for_acorn_plot(merged_info_hhblock_df)

In [ ]:
df_std_tou_mean

In [ ]:
df_acorn_mean

In [ ]:
df_acorn_grouped_mean

In [ ]:
plot_level(1, lambda: plot_average_daily_consumption(df_std_tou_mean, 'Std or ToU'))

In [ ]:
plot_level(1, lambda: plot_with_seasonal_colors(df_acorn_mean, 'Acorn'))

In [ ]:
plot_level(1, lambda: plot_average_daily_consumption(df_acorn_grouped_mean, 'Acorn Grouped'))

(不同富裕程度的周）

## Weather Data

temperatureMax - 当天的最高气温。
temperatureMaxTime - 最高气温发生的时间。
windBearing - 风向，表示风从哪个方向吹来，通常以角度表示。
icon - 天气情况的图标代码，如晴天、多云等。
dewPoint - 露点温度，表示空气达到饱和（露水开始凝结）的温度。
temperatureMinTime - 最低气温发生的时间。
cloudCover - 云量，表示天空被云层覆盖的比例。
windSpeed - 风速。
pressure - 大气压强。
apparentTemperatureMinTime - 体感最低温度发生的时间。
apparentTemperatureHigh - 当天的最高体感温度。
precipType - 降水类型，如雨、雪。
visibility - 能见度。
humidity - 湿度。
apparentTemperatureHighTime - 最高体感温度发生的时间。
apparentTemperatureLow - 当天的最低体感温度。
apparentTemperatureMax - 当天的最高体感温度。
uvIndex - 紫外线指数。
time - 观测时间。
sunsetTime - 日落时间。
temperatureLow - 当天的最低气温。
temperatureMin - 同temperatureLow，当天的最低气温。
temperatureHigh - 当天的最高气温。
sunriseTime - 日出时间。
temperatureHighTime - 最高气温发生的时间。
uvIndexTime - 紫外线指数达到最高点的时间。
summary - 天气概况的文字描述。
temperatureLowTime - 最低气温发生的时间。
apparentTemperatureMin - 当天的最低体感温度。
apparentTemperatureMaxTime - 最高体感温度发生的时间。
apparentTemperatureLowTime - 最低体感温度发生的时间。
moonPhase - 月相。

In [ ]:
weather_daily_darksky

In [ ]:
weather_daily_darksky.info()

In [ ]:
missing_weather_daily_darksky_values = weather_daily_darksky.isnull().sum()
missing_weather_daily_darksky_values

visibility - 能见度，通常以公里或英里为单位，表示在特定天气条件下最远可见的距离。
windBearing - 风向，表示风从哪个方向吹来。通常以角度表示，其中0度代表北风，90度代表东风，180度代表南风，270度代表西风。
temperature - 气温，通常以摄氏度或华氏度为单位，表示空气的热度。
time - 时间，数据采集的具体时间点，通常以UNIX时间戳或可读格式表示。
dewPoint - 露点温度，当空气冷却到无法容纳其中所有水蒸气时，水蒸气凝结成露水的温度。
pressure - 大气压强，表示空气的重量压在地面上的力量，通常以百帕斯卡（hPa）或毫巴（mb）为单位。
apparentTemperature - 体感温度，综合考虑风速、湿度和实际气温对人体感知温度的影响。
windSpeed - 风速，表示风的快慢，通常以每秒米数或每小时英里数表示。
precipType - 降水类型，如雨、雪、冰雹等。
icon - 天气图标的标识，用于直观表示天气状况，如晴天、多云、雨天等。
humidity - 湿度，表示空气中水蒸气的含量，通常以百分比表示。
summary - 天气概况的文字描述，提供对当前或预测天气状况的简短总结。

In [ ]:
weather_hourly_darksky.head(1000)

In [ ]:
# 确保time列为datetime格式
weather_hourly_darksky['time'] = pd.to_datetime(weather_hourly_darksky['time'])

In [ ]:
weather_hourly_darksky.info()

### 缺失值处理

In [ ]:
missing_weather_hourly_darksky_values = weather_hourly_darksky.isnull().sum()
missing_weather_hourly_darksky_values

In [ ]:
weather_hourly_darksky['pressure'] = weather_hourly_darksky['pressure'].ffill()

In [ ]:
weather_hourly_darksky.info()
missing_weather_hourly_darksky_values = weather_hourly_darksky.isnull().sum()
missing_weather_hourly_darksky_values

In [ ]:
def plot_weather_boxplot(df, column_name, time_type):
    # 确保time列为datetime格式
    if not pd.api.types.is_datetime64_any_dtype(df['time']):
        df['time'] = pd.to_datetime(df['time'])

    # 根据时间类型提取对应的时间单位
    if time_type == 'hour':
        df['time_unit'] = df['time'].dt.hour
    elif time_type == 'day':
        df['time_unit'] = df['time'].dt.date
    elif time_type == 'month':
        df['time_unit'] = df['time'].dt.month
    elif time_type == 'quarter':
        df['time_unit'] = df['time'].dt.quarter
    elif time_type == 'year':
        df['time_unit'] = df['time'].dt.year
    else:
        raise ValueError("Invalid time_type provided. Choose from 'hour', 'day', 'month', 'quarter', 'year'.")

    # 绘制箱线图
    plt.figure(figsize=(12, 8))
    sns.boxplot(x='time_unit', y=column_name, data=df)
    plt.title(f'Box plot of {column_name} over {time_type}')
    plt.xlabel(time_type.capitalize())
    plt.ylabel(column_name.capitalize())
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()


In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'temperature', 'hour'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'apparentTemperature', 'hour'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'temperature', 'day'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'apparentTemperature', 'day'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'temperature', 'month'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'apparentTemperature', 'month'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'temperature', 'quarter'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'apparentTemperature', 'quarter'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'temperature', 'year'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'apparentTemperature', 'year'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'visibility', 'hour'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'windBearing', 'month'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'dewPoint', 'month'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'pressure', 'month'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'humidity', 'hour'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'humidity', 'month'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'windSpeed', 'hour'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'windSpeed', 'quarter'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'precipType', 'month'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'icon', 'month'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'icon', 'quarter'))

In [ ]:
plot_level(2, lambda: plot_weather_boxplot(weather_hourly_darksky, 'summary', 'month'))

In [ ]:
def plot_weather_scatterplot(df, column_name1, column_name2):
    # 绘制散点图
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x=column_name1, y=column_name2, data=df)
    plt.title(f'Scatter plot of {column_name1} vs. {column_name2}')
    plt.xlabel(column_name1.capitalize())
    plt.ylabel(column_name2.capitalize())
    plt.grid(True)
    plt.show()

In [ ]:
plot_level(1, lambda: plot_weather_scatterplot(weather_hourly_darksky, 'temperature', 'apparentTemperature'))

In [ ]:
def plot_weather_lineplot(df, column_name, time_type):
    # 确保time列为datetime格式
    if not pd.api.types.is_datetime64_any_dtype(df['time']):
        df['time'] = pd.to_datetime(df['time'])

    # 根据时间类型提取对应的时间单位
    if time_type == 'hour':
        df['time_unit'] = df['time'].dt.hour
    elif time_type == 'day':
        df['time_unit'] = df['time'].dt.date
    elif time_type == 'month':
        df['time_unit'] = df['time'].dt.month
    elif time_type == 'quarter':
        df['time_unit'] = df['time'].dt.quarter
    elif time_type == 'year':
        df['time_unit'] = df['time'].dt.year
    else:
        raise ValueError("Invalid time_type provided. Choose from 'day', 'month', 'quarter', 'year'.")

    # 绘制折线图
    plt.figure(figsize=(12, 8))
    sns.lineplot(x='time_unit', y=column_name, data=df)
    plt.title(f'Line plot of {column_name} over {time_type}')
    plt.xlabel(time_type.capitalize())
    plt.ylabel(column_name.capitalize())
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()

In [ ]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'temperature', 'hour'))

In [ ]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'apparentTemperature', 'hour'))

In [ ]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'temperature', 'day'))

In [ ]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'apparentTemperature', 'day'))

In [ ]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'temperature', 'month'))

In [ ]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'apparentTemperature', 'month'))

In [ ]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'temperature', 'quarter'))

In [ ]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'apparentTemperature', 'quarter'))

In [ ]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'temperature', 'year'))

In [ ]:
plot_level(2, lambda: plot_weather_lineplot(weather_hourly_darksky, 'apparentTemperature', 'year'))

# 时间序列聚类分析
1. 数据预处理
1.1 同步数据时间粒度
家庭用电量数据：以半小时为单位，需要将其转换为小时单位以匹配天气数据的时间粒度。这可以通过取每小时内两个半小时用电量的平均值来实现。
天气数据：已经以小时为单位，直接使用。
1.2 时间对齐
确保两个数据集的时间戳对齐。这可能涉及到将时间戳转换为统一的格式（如UNIX时间戳或标准日期时间格式），并确保两个数据集覆盖相同的时间段。
1.3 数据合并
根据时间戳将家庭用电量数据和天气数据合并为一个数据集。每个时间点的数据应包括用电量和所有天气参数。
2. 特征工程
特征选择：基于数据理解，选择对聚类可能有影响的特征，例如，体感温度、湿度、风速可能会对用电量有直接影响。
特征转换：将所有特征标准化或归一化，以确保它们在相同的尺度上进行比较。
3. 相似性度量与聚类算法选择
相似性度量：考虑使用动态时间弯曲（DTW）作为相似性度量，因为它能够有效处理时间序列之间的时间偏移和伸缩。
聚类算法：可以考虑使用K-均值聚类（特别是如果使用DTW，那么是DTW的变体，如K-Shape聚类），或者层次聚类算法，后者不需要预先指定聚类数目。
4. 聚类执行
执行聚类算法，将时间序列数据分组为不同的聚类。这些聚类可能基于用电行为和天气条件的相似模式。
5. 聚类结果分析与解释
分析聚类结果：检查每个聚类的特征，如平均用电量、平均气温、平均湿度等，以理解不同聚类代表的用电行为和天气条件的模式。
结果可视化：使用时间序列图、雷达图或热图来展示聚类结果，这有助于直观理解不同聚类之间的差异。
6. 后续步骤
根据聚类结果，可以进一步分析特定天气条件下的用电模式，或者识别特定的用电行为模式对应的天气条件，这对于能源需求预测和优化能源供应具有重要意义。


## 数据预处理

1. 家庭用电量数据预处理
1.1 转换时间粒度
由于家庭用电量数据是以半小时为单位，而天气数据是以小时为单位，需要将家庭用电量数据转换为小时单位。这可以通过计算每小时内两个半小时用电量的平均值来实现。

In [ ]:
# 先复制原数据中的非用电量数据列
filtered_big_hhblock_one_hourly = filtered_big_hhblock[['LCLid', 'day']].copy()

# 计算每小时的平均用电量，并将结果添加到新DataFrame中
for i in range(24):
    filtered_big_hhblock_one_hourly[f'hh_{i}'] = filtered_big_hhblock[[f'hh_{2 * i}', f'hh_{2 * i + 1}']].sum(axis=1)

In [ ]:
filtered_big_hhblock_one_hourly.head(1000)

In [ ]:
filtered_big_hhblock_one_hourly.info()

2. 数据合并
2.1 时间格式转换
根据day和time列将两个数据集合并。这可能需要将家庭用电量数据的day列转换为与天气数据中time列相同的日期加小时的格式。

In [ ]:
@memory.cache
def turn_wide_to_long(filtered_big_hhblock_one_hourly):
    # 转换LCLid列为分类数据类型
    filtered_big_hhblock_one_hourly['LCLid'] = filtered_big_hhblock_one_hourly['LCLid'].astype('category')
    
    # 转换Pandas DataFrame为Dask DataFrame
    ddf = dd.from_pandas(filtered_big_hhblock_one_hourly, npartitions=10)  # 根据数据大小和内存情况调整分区数

    # 先将Dask DataFrame转换为Pandas DataFrame进行melt操作
    pdf = ddf.compute()  # 将Dask DataFrame转换回Pandas DataFrame

    # 使用Pandas的melt函数进行操作
    pdf_melted = pdf.melt(id_vars=['LCLid', 'day'], var_name='hour', value_name='energy')

    # 转换 'hour' 列，从 'hh_X' 提取小时数，并将其转换为整数
    pdf_melted['hour'] = pdf_melted['hour'].str.extract('(\d+)').astype(int)

    # 假设 'day' 已经是 datetime 类型，如果不是，先转换它
    # 将 'day' 和 'hour' 合并成一个 datetime 列
    pdf_melted['day-hour'] = pd.to_datetime(pdf_melted['day']) + pd.to_timedelta(pdf_melted['hour'], unit='h')

    # 删除不再需要的列
    pdf_melted.drop(['day', 'hour'], axis=1, inplace=True)

    # 再将处理后的Pandas DataFrame转换回Dask DataFrame
    ddf_melted = dd.from_pandas(pdf_melted, npartitions=10)

    # 设置进度条
    pbar = ProgressBar()
    pbar.register()

    df_result = ddf_melted.compute()  # Progress bar will show up here

    # 注销进度条
    pbar.unregister()

    # 现在df_result是一个Pandas DataFrame，包含了处理后的数据

    df_result.to_parquet('filtered_big_hhblock_one_hourly_one_row.parquet')

    return df_result

In [ ]:
# %%time
# if os.path.exists('filtered_big_hhblock_one_hourly_one_row.parquet'):
#     filtered_big_hhblock_one_hourly_one_row = pd.read_parquet('filtered_big_hhblock_one_hourly_one_row.parquet')
# else:
#     filtered_big_hhblock_one_hourly_one_row = turn_wide_to_long(filtered_big_hhblock_one_hourly)

In [ ]:
%%time
filtered_big_hhblock_one_hourly_one_row = turn_wide_to_long(filtered_big_hhblock_one_hourly)

In [ ]:
filtered_big_hhblock_one_hourly_one_row.head(1000)

2.2 数据合并，将转换格式后的家庭用电量数据和天气数据合并为一个数据集

In [ ]:
# sampled_filtered_big_hhblock_one_hourly_one_row = filtered_big_hhblock_one_hourly_one_row.sample(frac=0.001, random_state=0)

In [ ]:
# sampled_filtered_big_hhblock_one_hourly_one_row

In [ ]:
weather_hourly_darksky.head(1000)

In [ ]:
weather_hourly_darksky.info()

In [ ]:
@memory.cache
def combine_data_with_mapping(filtered_big_hhblock_one_hourly_one_row, weather_hourly_darksky):
    # 初始化一个字典来保存转换映射
    mapping_dict = {}

    # 转换为dask dataframe
    ddf_weather = dd.from_pandas(weather_hourly_darksky, npartitions=10)

    # 首先全局计算出现频率并生成映射字典
    for col in ['precipType', 'icon', 'summary']:
        if col in ddf_weather.columns:
            # 使用dask计算每个值的出现次数
            frequencies = ddf_weather[col].value_counts().compute()
            # 根据频率排序，生成映射
            mapping = {value: i for i, value in enumerate(frequencies.sort_values(ascending=False).index)}
            mapping_dict[col] = mapping

            # 应用映射到每个分区，并提供meta参数以避免警告
        ddf_weather[col] = ddf_weather[col].map(mapping, meta=('x', 'int64')).astype('int64')
        
    # 使用Dask进行数据合并处理
    ddf_result = dd.from_pandas(filtered_big_hhblock_one_hourly_one_row, npartitions=10)

    combined_ddf = dd.merge(ddf_result, ddf_weather, left_on='day-hour', right_on='time', how='inner')

    # 设置进度条
    pbar = ProgressBar()
    pbar.register()

    # 计算合并后的Dask DataFrame，转换为Pandas DataFrame
    combined_pd = combined_ddf.compute()

    # 注销进度条
    pbar.unregister()

    combined_pd.drop(['time'], axis=1, inplace=True)

    combined_pd.to_parquet('combined_hourly_weather.parquet')

    # 返回合并后的DataFrame和映射字典
    return combined_pd, mapping_dict


In [ ]:
# %%time
# if os.path.exists('combined_hourly_weather.parquet'):
#     print('Reading combined_hourly_weather.parquet...')
#     combined_hourly_weather = pd.read_parquet('combined_hourly_weather.parquet')
# else:
#     print('Combining data...')
#     combined_hourly_weather, mapping_dict = combine_data_with_mapping(filtered_big_hhblock_one_hourly_one_row, weather_hourly_darksky)

In [ ]:
%%time
combined_hourly_weather, mapping_dict = combine_data_with_mapping(filtered_big_hhblock_one_hourly_one_row, weather_hourly_darksky)

In [ ]:
combined_hourly_weather.head(1000)

In [ ]:
display(mapping_dict)

In [ ]:
combined_hourly_weather.head(1000)

2.3 缺失值处理

In [ ]:
missing_values_count = combined_hourly_weather.isnull().sum()
missing_values_count

2.4 数据标准化

2.4.1 特征列生成

将整体的时间转换为拆分的时间特征，以便更好地表示时间的周期性。
'day-hour' -> 'hour', 'day', 'day_of_week', 'month', 'quarter', 'year'
再将这些周期性时间特征转换为循环特征，以便更好地在模型中表示时间的周期性。
'hour' -> 'hour_sin', 'hour_cos'
'day' -> 'day_sin', 'day_cos'
'day_of_week' -> 'day_of_week_sin', 'day_of_week_cos'
'month' -> 'month_sin', 'month_cos'
'quarter' -> 'quarter_sin', 'quarter_cos'
'year' -> 'year_sin', 'year_cos'

In [ ]:
@memory.cache
def add_cyclic_features(df):
    # 从'day-hour'列提取周期性特征
    df['hour'] = df['day-hour'].dt.hour
    df['day'] = df['day-hour'].dt.day
    df['day_of_week'] = df['day-hour'].dt.dayofweek
    df['month'] = df['day-hour'].dt.month
    df['quarter'] = df['day-hour'].dt.quarter
    df['year'] = df['day-hour'].dt.year

    # 转换小时特征为循环特征
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

    # # 转换天为循环特征
    df['day_sin'] = np.sin(2 * np.pi * df['day'] / 31)
    df['day_cos'] = np.cos(2 * np.pi * df['day'] / 31)

    # 转换星期特征为循环特征
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    # 转换月份特征为循环特征
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # 转换季度特征为循环特征
    df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
    df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)

    # 转换年份特征为循环特征
    df['year_sin'] = np.sin(2 * np.pi * df['year'] / 365)
    df['year_cos'] = np.cos(2 * np.pi * df['year'] / 365)
    
    # 删除原始时间特征列
    df.drop(['day-hour', 'hour', 'day', 'day_of_week', 'month', 'quarter', 'year'], axis=1, inplace=True)

    df.to_parquet('cluster_cyclic_features.parquet')
    
    return df

In [ ]:
%%time
if os.path.exists('cluster_cyclic_features.parquet'):
    print('Reading cluster_cyclic_features.parquet...')
    combined_hourly_weather_all_features = pd.read_parquet('cluster_cyclic_features.parquet')
else:
    print('Adding cyclic features...')
    combined_hourly_weather_all_features = add_cyclic_features(combined_hourly_weather)

In [ ]:
# %%time
# combined_hourly_weather_all_features = add_cyclic_features(combined_hourly_weather)

In [ ]:
combined_hourly_weather_all_features.head(1000)

In [ ]:
combined_hourly_weather_all_features.columns

2.4.2 计算相关系数

In [ ]:
def compute_correlation(series_energy, column, series_column):
    # 确保序列是NumPy数组格式
    series_energy_np = np.array(series_energy)
    series_column_np = np.array(series_column)

    # 计算Pearson、Spearman和Kendall相关性
    pearson_corr = series_energy.corr(series_column, method='pearson')
    spearman_corr = series_energy.corr(series_column, method='spearman')
    kendall_corr = series_energy.corr(series_column, method='kendall')

    # 计算DTW距离
    dtw_distance = dtw.distance(series_energy_np, series_column_np)

    return column, pearson_corr, spearman_corr, kendall_corr, dtw_distance

In [ ]:
# sampled_combined_hourly_weather_all_features = combined_hourly_weather_all_features.sample(frac=0.0001, random_state=0)
# sampled_combined_hourly_weather_all_features

In [ ]:
# series_energy = sampled_combined_hourly_weather_all_features['energy']

In [ ]:
# df_dropped = sampled_combined_hourly_weather_all_features.drop(['LCLid', 'energy'], axis=1)

In [ ]:
# results = Parallel(n_jobs=32)(delayed(compute_correlation)(series_energy, column, df_dropped[column]) for column in df_dropped.columns)

In [ ]:
# correlation_df = pd.DataFrame(results, columns=['Column', 'Pearson', 'Spearman', 'Kendall', 'DTW']).set_index('Column')

In [ ]:
# correlation_df

In [ ]:
# print(correlation_df)

某次frac=0.0001(8254 rows)的结果：
                      Pearson  Spearman   Kendall           DTW
Column                                                         
visibility          -0.006904  0.016826  0.011255    443.862792
windBearing         -0.006027 -0.006485 -0.004370  18906.127867
temperature         -0.083022 -0.054545 -0.036281    626.412410
dewPoint            -0.097926 -0.078054 -0.051959    466.723752
pressure            -0.011370 -0.005006 -0.003272  91988.135263
apparentTemperature -0.086513 -0.055480 -0.036885    683.244370
windSpeed            0.013820  0.030065  0.020107    190.000330
precipType           0.034258  0.018888  0.015430     48.576321
icon                -0.012922 -0.050057 -0.036609    120.986451
humidity            -0.011574 -0.054400 -0.036372     43.448652
summary             -0.019059 -0.054705 -0.040448    116.506113
hour_sin            -0.163699 -0.234208 -0.160950     64.558582
hour_cos            -0.029295 -0.063006 -0.042857     64.119046
day_sin              0.004296 -0.001687 -0.001163     64.604965
day_cos             -0.007818  0.000127  0.000092     63.945644
day_of_week_sin     -0.008611 -0.005341 -0.003856     63.926037
day_of_week_cos      0.024850  0.011692  0.008394     65.651662
month_sin            0.069973  0.042131  0.029486     65.454581
month_cos            0.084735  0.067386  0.047056     63.750382
quarter_sin          0.095618  0.057290  0.042831     64.398062
quarter_cos          0.040359  0.061399  0.045760     60.926613
year_sin            -0.024352 -0.008327 -0.006598     65.629302
year_cos             0.024858  0.008327  0.006598    137.013980



In [ ]:
# sampled_combined_hourly_weather_all_features.columns

In [ ]:
# columns_morethan0 = []
# for row in correlation_df.itertuples():
#     if (row.Pearson > 0) or (row.Spearman > 0) or (row.Kendall > 0) or (row.DTW < 100):
#         columns_morethan0.append(row.Index)
#         
# print(columns_morethan0)

In [ ]:
# # 将DataFrame的列名和columns_morethan0转换为集合
# columns_set = set(sampled_combined_hourly_weather_all_features.columns)
# columns_morethan0_set = set(columns_morethan0)
# 
# # 检查是否所有列都在columns_morethan0中
# if not columns_set.issubset(columns_morethan0_set):
#     missing_columns = columns_set - columns_morethan0_set
#     print("These columns are not in columns_morethan0:", missing_columns)


sin和cos只有一个循环特征相关性高的可能原因：
- 非对称影响：如果目标变量对时间的敏感度是非对称的，比如说，某些现象或行为在一天中的某些时间段更为频繁或强烈，这种非对称性可能会导致sin或cos中的一个与目标变量有更高的相关性。例如，如果energy消耗在夜间减少但在白天变化不大，那么sin函数（代表了一天中的变化）可能与energy有更高的相关性。
- 数据的内在特性：数据集中的某些特性可能与特定的循环特征更加对齐。例如，如果能量消耗主要在某个特定的时间段内变化显著，那么这种模式可能会更多地反映在sin或cos值的变化上。

根据结果，排除相关性较低的特征：'dewPoint', 'pressure', 'icon', 'summary', 同时，虽然'temperature', 'apparentTemperature'的各项相关性都不高，但是由于它们是气象数据中的重要特征，因此保留。

In [ ]:
cluster_columns_use = [
    'LCLid', 'energy',
    'hour_sin', 'hour_cos',
    'day_sin', 'day_cos',
    'day_of_week_sin', 'day_of_week_cos',
    'month_sin', 'month_cos',
    'quarter_sin', 'quarter_cos',
    'year_sin', 'year_cos',
    'visibility', 
    'windSpeed', 
    'precipType',
    'humidity',
    'temperature', 'apparentTemperature',]
cluster_columns_not_use = ['dewPoint', 'pressure', 'icon', 'summary', 'windBearing' ]

2.4.3 数值型数据标准化

In [ ]:
df_cluster = combined_hourly_weather_all_features[cluster_columns_use]
df_cluster.head(1000)

In [ ]:
def std_scaler(df):
    # 初始化StandardScaler
    scaler = StandardScaler()
    
    # 需要进行标准化的列，sin，cos列不需要标准化，因为它们已经在0到1之间，precipType是分类数据，也不需要标准化
    norminal_columns = ['energy', 'visibility', 'windSpeed', 'humidity', 'temperature', 'apparentTemperature']
    
    df_copy = df.copy()
    
    # 直接在原地修改，节省空间（注意：这会修改原始DataFrame）
    df_copy[norminal_columns] = scaler.fit_transform(df_copy[norminal_columns])

    df_copy.to_parquet('cluster_cyclic_features_normalised.parquet')
    
    return df_copy

In [ ]:
%%time
if os.path.exists('cluster_cyclic_features_normalised.parquet'):
    print('Reading cluster_cyclic_features_normalised.parquet...')
    df_cluster_normalised = pd.read_parquet('cluster_cyclic_features_normalised.parquet')
else:
    print('Normalising data...')
    df_cluster_normalised = std_scaler(df_cluster)

In [ ]:
# df_cluster_normalised = std_scaler(df_cluster)

In [ ]:
df_cluster_normalised.head(1000)

In [ ]:
df_cluster_normalised.info()

## 3 时间序列MiniBatchKMeans聚类

In [ ]:
sampled_df_cluster_normalised = df_cluster_normalised.sample(frac=0.00001, random_state=0)

In [ ]:
sampled_df_cluster_normalised

In [ ]:
features = {
    'hour': ['energy', 'hour_sin', 'hour_cos'],
    'day': ['energy', 'day_sin', 'day_cos'],
    'day_of_week': ['energy', 'day_of_week_sin', 'day_of_week_cos'],
    'month': ['energy', 'month_sin', 'month_cos'],
    'quarter': ['energy', 'quarter_sin', 'quarter_cos'],
    'year': ['energy', 'year_sin', 'year_cos'],
    'visibility': ['energy', 'visibility'],
    'windSpeed': ['energy', 'windSpeed'],
    'precipType': ['energy', 'precipType'],
    'humidity':['energy', 'humidity'],
    'weather': ['energy', 'visibility',  'windSpeed', 'precipType', 'humidity'],
    'temperature': ['energy', 'temperature', 'apparentTemperature']}

### 3.1 分别对单一因素进行聚类

In [ ]:
# !pip install fastdtw
# !pip install umap-learn

In [ ]:
# def calculate_umap(df, distance_matrix):
#     umap = UMAP(n_components=2, random_state=42)
#     data_transformed = umap.fit_transform(distance_matrix)
#     return data_transformed
# 
# def fit_minibatchkmeans(data_transformed, n_clusters):
#     mbkmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=100, random_state=42, n_init=3)
#     labels = mbkmeans.fit_predict(data_transformed)
#     return mbkmeans, labels
# 
# def cluster_features(df, features, n_clusters=5, mode='hyper'):
#     # 计算 DTW 距离矩阵
#     distance_matrix = np.zeros((len(df), len(df)))
#     for i in tqdm(range(len(df)), desc='Calculating Distance Matrix'):
#         for j in range(i, len(df)):  # 利用距离矩阵的对称性
#             distance, _ = fastdtw(df[features].iloc[i], df[features].iloc[j])
#             distance_matrix[i, j] = distance_matrix[j, i] = distance
# 
#     # 并行执行UMAP降维和MiniBatchKMeans聚类
#     with tqdm(desc='Processing', total=2, position=0, leave=True) as pbar:
#         data_transformed = Parallel(n_jobs=32)(delayed(calculate_umap)(df, distance_matrix) for _ in range(1))
#         pbar.update(1)
# 
#         mbkmeans, labels = Parallel(n_jobs=32)(delayed(fit_minibatchkmeans)(data_transformed[0], n_clusters) for _ in range(1))
#         pbar.update(1)
#     
#     if mode == 'hyper':  # 超参数搜索模式
#         inertia = mbkmeans.inertia_  # 计算 inertia 值
# 
#         # 计算轮廓系数
#         silhouette_avg = silhouette_score(data_transformed[0], labels[0]) if len(set(labels[0])) > 1 else 0
# 
#         return labels[0], silhouette_avg, data_transformed[0], inertia
#     elif mode == 'cluster':  # 聚类模式
#         return labels[0], data_transformed[0]
#     else:
#         raise ValueError("Invalid mode. Choose from 'hyper' or 'cluster'.")


In [ ]:
def compute_distance(i, j, df, features):
    distance, _ = fastdtw(df[features].iloc[i], df[features].iloc[j])
    return distance

def compute_distance_matrix(df, features):
    n_samples = len(df)
    distance_matrix = np.zeros((n_samples, n_samples))

    # 并行计算距离矩阵
    # 并行计算距离矩阵，并结合tqdm显示进度条
    results = []
    with tqdm(desc='Processing fastdtw', total=n_samples*(n_samples+1)//2) as pbar:
        for i in range(n_samples):
            results.extend(Parallel(n_jobs=-1)(delayed(compute_distance)(i, j, df, features)
                                               for j in range(i, n_samples)))
            pbar.update(n_samples - i)

    # 填充距离矩阵
    idx = 0
    for i in range(n_samples):
        for j in range(i, n_samples):
            distance_matrix[i, j] = distance_matrix[j, i] = results[idx]
            idx += 1

    return distance_matrix

def calculate_umap(df, distance_matrix):
    umap = UMAP(n_components=2, random_state=42)
    data_transformed = umap.fit_transform(distance_matrix)
    return data_transformed

def fit_minibatchkmeans(data_transformed, n_clusters):
    mbkmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=100, random_state=42, n_init=3)
    labels = mbkmeans.fit_predict(data_transformed)
    return mbkmeans, labels

def cluster_features(df, features, n_clusters=5, mode='hyper'):
    # 计算 DTW 距离矩阵
    distance_matrix = compute_distance_matrix(df, features)

    # 并行执行UMAP降维和MiniBatchKMeans聚类
    with tqdm(desc='Processing UMAP and MiniBatchKMeans', total=2, position=0, leave=True) as pbar:
        data_transformed = Parallel(n_jobs=32)(delayed(calculate_umap)(df, distance_matrix) for _ in range(1))
        pbar.update(1)

        mbkmeans, labels = Parallel(n_jobs=32)(delayed(fit_minibatchkmeans)(data_transformed[0], n_clusters) for _ in range(1))
        pbar.update(1)

    if mode == 'hyper':  # 超参数搜索模式
        inertia = mbkmeans.inertia_  # 计算 inertia 值

        # 计算轮廓系数
        silhouette_avg = silhouette_score(data_transformed[0], labels[0]) if len(set(labels[0])) > 1 else 0

        return labels[0], silhouette_avg, data_transformed[0], inertia
    elif mode == 'cluster':  # 聚类模式
        return labels[0], data_transformed[0]
    else:
        raise ValueError("Invalid mode. Choose from 'hyper' or 'cluster'.")


In [ ]:
# def cluster_features(df, features, n_clusters=5, mode='hyper'):
#     # 计算 DTW 距离矩阵
#     distance_matrix = np.zeros((len(df), len(df)))
#     for i in tqdm(range(len(df)), desc='Calculating Distance Matrix'):
#         for j in range(i, len(df)):  # 利用距离矩阵的对称性
#             distance, _ = fastdtw(df[features].iloc[i], df[features].iloc[j])
#             distance_matrix[i, j] = distance_matrix[j, i] = distance
# 
#     # 降维
#     umap = UMAP(n_components=2, random_state=42)
#     data_transformed = umap.fit_transform(distance_matrix)
# 
#     # 聚类
#     mbkmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=100, random_state=42, n_init=3)
#     labels = mbkmeans.fit_predict(data_transformed)
#     if mode == 'hyper':  # 超参数搜索模式
#         inertia = mbkmeans.inertia_  # 计算 inertia 值
# 
#         # 计算轮廓系数
#         silhouette_avg = silhouette_score(data_transformed, labels) if len(set(labels)) > 1 else 0
# 
#         return labels, silhouette_avg, data_transformed, inertia
#     elif mode == 'cluster':  # 聚类模式
#         return labels, data_transformed
#     else:
#         raise ValueError("Invalid mode. Choose from 'hyper' or 'cluster'.")


In [ ]:
def find_optimal_clusters(df, features, m, n):
    silhouette_avgs = []
    inertia_values = []

    # 尝试不同的聚类数
    for n_clusters in tqdm(range(m, n), desc='Finding optimal clusters'):
        _, silhouette_avg, data_transformed, inertia = cluster_features(df, features, n_clusters=n_clusters)
        silhouette_avgs.append(silhouette_avg)
        inertia_values.append(inertia)

        # 打印相关信息
        tqdm.write(f"Number of clusters: {n_clusters}, Silhouette Score: {silhouette_avg}, Inertia: {inertia}")
        
    return silhouette_avgs, inertia_values

In [ ]:
def plot_elbow_and_silhouette(inertia_values, silhouette_avgs, m, n):
    # 绘制肘部法则图和轮廓系数折线图
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(range(m, n), inertia_values, marker='o')
    plt.title('Elbow Method')
    plt.xlabel('Number of clusters')
    plt.ylabel('Inertia')

    # 绘制轮廓系数折线图
    plt.subplot(1, 2, 2)
    plt.plot(range(m, n), silhouette_avgs, marker='o')
    plt.title('Silhouette Score')
    plt.xlabel('Number of clusters')
    plt.ylabel('Silhouette Score')

    plt.tight_layout()
    plt.show()


In [ ]:
def plot_clusters(data_transformed, labels, title):
    # 绘制聚类结果
    plt.figure(figsize=(12, 6))
    plt.scatter(data_transformed[:, 0], data_transformed[:, 1], c=labels, cmap='viridis', alpha=0.5)
    plt.colorbar()
    plt.title(title)
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')
    plt.show()

#### 3.1.1 以小时为单位聚类

In [ ]:
labels, silhouette_avg, data_transformed, inertia = cluster_features(sampled_df_cluster_normalised, features['hour'], n_clusters=5, mode='hyper')

In [ ]:
print(f'Silhouette Score: {silhouette_avg:.2f}')
print(f'Inertia: {inertia:.2f}')
# print('Labels:', labels)
# print('Data Transformed:', data_transformed)

In [ ]:
plot_clusters(data_transformed, labels, 'K-Means Clustering for Hour Features')

In [ ]:
%%time
silhouette_avgs, inertia_values = find_optimal_clusters(sampled_df_cluster_normalised, features['hour'], 2, 5)

In [ ]:
plot_elbow_and_silhouette(inertia_values, silhouette_avgs, 2, 5)

#### 3.1.2 以天为单位聚类

In [ ]:
# labels, silhouette_avg, data_transformed, inertia = cluster_features(sampled_df_cluster_normalised, features['day'], n_clusters=5, mode='hyper')

随机森林

In [ ]:
# from sklearn.model_selection import train_test_split
# 
# # 假设你的DataFrame名为df
# X = judge[['temperature']]  # 选择温度作为特征变量
# y = judge['energy']  # 能耗作为目标变量
# 
# # 划分训练集和测试集
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# from sklearn.ensemble import RandomForestRegressor
# 
# # 初始化随机森林回归器
# rf = RandomForestRegressor(n_estimators=100, random_state=42)  # n_estimators是树的数量
# 
# # 训练模型
# rf.fit(X_train, y_train)


In [ ]:
# from sklearn.metrics import mean_squared_error
# 
# # 对测试集进行预测
# y_pred = rf.predict(X_test)
# 
# # 计算并打印均方误差(MSE)作为性能的评估
# mse = mean_squared_error(y_test, y_pred)
# print(f"Mean Squared Error: {mse}")


In [ ]:
# # 获取特征重要性
# importances = rf.feature_importances_
# print(f"Importance of temperature: {importances[0]}")


In [ ]:
# # 对一系列温度值进行预测以绘制曲线
# temperature_range = np.linspace(X_train.min(), X_train.max(), 100).reshape(-1, 1)
# energy_pred = rf.predict(temperature_range)
# 
# # 绘制散点图和预测曲线
# plt.scatter(X_train, y_train, color='gray', alpha=0.5, label='Actual')
# plt.plot(temperature_range, energy_pred, color='red', label='Prediction')
# plt.xlabel('Temperature')
# plt.ylabel('Energy Consumption')
# plt.title('Energy Consumption vs Temperature')
# plt.legend()
# plt.show()
